# 07 - Full Repo Route Comparison SER

Notebook này gom **3 route** vào cùng một benchmark để so sánh công bằng:

| Route | Chạy gì | Mục đích |
|---|---|---|
| [1. Emonity-style](#route-1-emonity-style) | Parse metadata 4 dataset, extract feature riêng cho 1D/2D, train `1D-CNN`, `2D-CNN`, `CNN-BiLSTM`, rồi validation-weighted ensemble | Tái tạo ý tưởng repo Emonity trong cùng split |
| [2. CNN/LSTM/CLSTM-style](#route-2-cnn-lstm-clstm-style) | Load audio `duration=2.5`, `offset=0.6`, extract `ZCR + RMS/RMSE + MFCC flatten`, augment train bằng `noise`, `pitch`, `pitch+noise`, train `CNN`, `LSTM`, `CNN-LSTM/CLSTM` | Tái tạo pipeline repo CNN/LSTM/CLSTM trong cùng split |
| [3. Our 06F](#route-3-our-06f-domain-aware-mixture-adapter) | Temporal + spectral + stats + optional emotion2vec, co-attention, domain-aware adapters, auxiliary heads, validation selective fusion | Mô hình của project để so với 2 baseline |

## Nguyên tắc so sánh

**Split được tạo trước, dùng lại cho mọi route.**  
Augmentation chỉ được áp dụng trên `train`, không augment `validation/test`, để tránh leakage.

Notebook tạo các protocol:

1. `combined_random_60_20_20_all4`: gộp tất cả dataset, random split 60/20/20.
2. `combined_strict_speaker_all4`: gộp tất cả dataset, split theo speaker, không trùng speaker giữa train/val/test.
3. `single_random_60_20_20_<dataset>`: từng dataset riêng.
4. `single_strict_speaker_<dataset>`: từng dataset riêng, strict theo speaker nếu đủ speaker.

## Cách chạy nhanh

- Chạy các cell **Setup → Load Metadata → Shared Splits → Shared Utils**.
- Sau đó bấm vào route cần chạy:
  - [Run Route 1](#route-1-emonity-style)
  - [Run Route 2](#route-2-cnn-lstm-clstm-style)
  - [Run Route 3](#route-3-our-06f-domain-aware-mixture-adapter)
- Cuối cùng chạy [Leaderboard + Package Outputs](#leaderboard-and-package-outputs).

Có thể cấu hình bằng environment variable:

```bash
RUN_ROUTES=1,2,3
RUN_PROTOCOLS=combined_random_60_20_20_all4,combined_strict_speaker_all4
RUN_SINGLE_DATASET_PROTOCOLS=1
MAX_EPOCHS=35
QUICK_RUN=0
```

## 0. Notes From Reference Repos

Notebook này là **repo-faithful but fair comparison**:

- Giữ ý tưởng preprocessing/model chính của repo.
- Nhưng **không dùng split riêng của từng repo**, vì như vậy không công bằng.
- Tất cả route dùng cùng `metadata`, cùng label space 6 emotion, cùng split manifest.
- Mọi output được lưu vào `07_Full_Route_Comparison_outputs/`.

Reference routes:

- Emonity: 4 dataset CREMA-D/RAVDESS/TESS/SAVEE, MFCC/log-Mel/spectral/chroma/ZCR/RMS, 1D-CNN/2D-CNN/CNN-BiLSTM/ensemble.
- CNN/LSTM/CLSTM: 4 dataset CREMA-D/RAVDESS/SAVEE/TESS, ZCR/RMS/MFCC flatten, augmentation noise/pitch/time, CNN/LSTM/CLSTM.
- Our 06F: lấy từ notebook `06F_Domain_Aware_Mixture_Adapter_SER.ipynb`.

## 1. Setup

In [ ]:
from pathlib import Path
import os, json, time, random, zipfile, warnings, math, shutil
from collections import defaultdict

import numpy as np
import pandas as pd
import librosa
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    confusion_matrix, classification_report
)

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

SEED = int(os.getenv("SEED", "42"))
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
GPU_COUNT = torch.cuda.device_count()

PROJECT_ROOT = Path("/kaggle/working/Speech_Project") if Path("/kaggle/working").exists() else Path.cwd().resolve()
OUTPUT_DIR = PROJECT_ROOT / "07_Full_Route_Comparison_outputs"
CACHE_DIR = OUTPUT_DIR / "cache"
REPORT_DIR = OUTPUT_DIR / "reports"
FIGURE_DIR = OUTPUT_DIR / "figures"
MODEL_DIR = OUTPUT_DIR / "models"
PRED_DIR = OUTPUT_DIR / "predictions"
SPLIT_DIR = OUTPUT_DIR / "splits"

for d in [OUTPUT_DIR, CACHE_DIR, REPORT_DIR, FIGURE_DIR, MODEL_DIR, PRED_DIR, SPLIT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

COMMON_EMOTIONS = ["neutral", "happy", "sad", "angry", "fear", "disgust"]
LABEL_TO_ID = {e: i for i, e in enumerate(COMMON_EMOTIONS)}
ID_TO_LABEL = {i: e for e, i in LABEL_TO_ID.items()}
NUM_CLASSES = len(COMMON_EMOTIONS)

TARGET_SR = int(os.getenv("TARGET_SR", "16000"))
TARGET_DURATION = float(os.getenv("TARGET_DURATION", "4.0"))
TARGET_LENGTH = int(TARGET_SR * TARGET_DURATION)

# Route 1: Emonity-style
EMONITY_SR = int(os.getenv("EMONITY_SR", str(TARGET_SR)))
EMONITY_DURATION = float(os.getenv("EMONITY_DURATION", str(TARGET_DURATION)))
EMONITY_TARGET_FRAMES = int(os.getenv("EMONITY_TARGET_FRAMES", "128"))
EMONITY_AUGMENT_TRAIN = os.getenv("EMONITY_AUGMENT_TRAIN", "1") == "1"
EMONITY_AUG_COPIES = int(os.getenv("EMONITY_AUG_COPIES", "1"))

# Route 2: CNN/LSTM/CLSTM-style
CLSTM_SR = int(os.getenv("CLSTM_SR", "22050"))
CLSTM_DURATION = float(os.getenv("CLSTM_DURATION", "2.5"))
CLSTM_OFFSET = float(os.getenv("CLSTM_OFFSET", "0.6"))
CLSTM_AUGMENT_TRAIN = os.getenv("CLSTM_AUGMENT_TRAIN", "1") == "1"

# Route 3: Our 06F
OURS_USE_EMOTION2VEC = os.getenv("OURS_USE_EMOTION2VEC", "0") == "1"
OURS_E2V_DIM = int(os.getenv("OURS_E2V_DIM", "256"))
OURS_USE_BALANCED_SAMPLER = os.getenv("OURS_USE_BALANCED_SAMPLER", "1") == "1"

QUICK_RUN = os.getenv("QUICK_RUN", "0") == "1"
QUICK_RUN_SAMPLES_PER_DATASET = int(os.getenv("QUICK_RUN_SAMPLES_PER_DATASET", "180"))

MAX_EPOCHS = int(os.getenv("MAX_EPOCHS", "35"))
PATIENCE = int(os.getenv("PATIENCE", "8"))
BATCH_SIZE = int(os.getenv("BATCH_SIZE", "64"))
NUM_WORKERS = int(os.getenv("NUM_WORKERS", "0"))
LR = float(os.getenv("LR", "0.001"))
WEIGHT_DECAY = float(os.getenv("WEIGHT_DECAY", "0.0005"))

RUN_ROUTES = [x.strip() for x in os.getenv("RUN_ROUTES", "1,2,3").split(",") if x.strip()]
RUN_PROTOCOLS = [x.strip() for x in os.getenv(
    "RUN_PROTOCOLS",
    "combined_random_60_20_20_all4,combined_strict_speaker_all4"
).split(",") if x.strip()]
RUN_SINGLE_DATASET_PROTOCOLS = os.getenv("RUN_SINGLE_DATASET_PROTOCOLS", "1") == "1"

print("DEVICE:", DEVICE, "GPU_COUNT:", GPU_COUNT)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("QUICK_RUN:", QUICK_RUN)
print("RUN_ROUTES:", RUN_ROUTES)
print("RUN_PROTOCOLS:", RUN_PROTOCOLS)
print("RUN_SINGLE_DATASET_PROTOCOLS:", RUN_SINGLE_DATASET_PROTOCOLS)

## 2. Load Metadata And Audio Paths

Notebook ưu tiên dataset đã xử lý sẵn theo format project:

```text
ser_processed/
  metadata.csv
  audio_16k/
```

`metadata.csv` nên có các cột như `sample_id`, `dataset`, `emotion`, `speaker_global` hoặc `speaker_id`, `filepath`.

**Audio source priority:** nếu output 0102update có `audio_16k_raw` hoặc `audio_16k_clean_full`, notebook sẽ ưu tiên các folder này để route Emonity/CNN-LSTM-CLSTM gần raw-audio pipeline hơn. `audio_16k` chỉ là fallback cho output cũ.

In [ ]:
def find_processed_root():
    env_path = os.getenv("SER_PROCESSED", "").strip()
    candidates = []
    if env_path:
        candidates.append(Path(env_path))
    candidates.extend([
        Path("/kaggle/input/ser-processed/ser_processed"),
        Path("/kaggle/input/ser-processed-update/ser_processed_update"),
        Path("/kaggle/input/ser-processed"),
        Path("/kaggle/input/ser_processed"),
        Path("/kaggle/working/ser_processed"),
        Path("/kaggle/working/ser_processed_update"),
        PROJECT_ROOT / "ser_processed",
        PROJECT_ROOT / "ser_processed_update",
        Path.cwd() / "ser_processed",
        Path.cwd() / "ser_processed_update",
        Path.cwd() / "01&02_Data_and_DataProcessing" / "ser_processed",
        Path.cwd() / "01&02_Data_and_DataProcessing" / "ser_processed_update",
    ])
    for root in candidates:
        if (root / "metadata.csv").exists():
            return root
    for base in [Path("/kaggle/input"), Path("/kaggle/working"), Path.cwd(), PROJECT_ROOT]:
        if base.exists():
            for meta in base.rglob("metadata.csv"):
                parent = meta.parent
                if "ser" in str(parent).lower() and (parent / "metadata.csv").exists():
                    return parent
    raise FileNotFoundError("Cannot find ser_processed-like directory with metadata.csv")

PROCESSED_ROOT = find_processed_root()
metadata = pd.read_csv(PROCESSED_ROOT / "metadata.csv")

audio_dir_candidates = [
    # Prefer less-destructive audio for full repo routes.
    PROCESSED_ROOT / "audio_16k_raw",
    PROCESSED_ROOT / "audio_16k_clean_full",
    PROCESSED_ROOT / "audio",
    PROCESSED_ROOT / "wav",
    # Fallback for older ser_processed packages.
    PROCESSED_ROOT / "audio_16k",
]
AUDIO_DIR = next((p for p in audio_dir_candidates if p.exists()), None)

def infer_audio_path(row):
    sample_id = str(row.get("sample_id", ""))
    if AUDIO_DIR is not None and sample_id:
        p = AUDIO_DIR / f"{sample_id}.wav"
        if p.exists():
            return str(p)
    for col in ["audio_path", "processed_path", "filepath", "path", "file"]:
        if col in row and pd.notna(row[col]):
            p = Path(str(row[col]))
            if p.exists():
                return str(p)
            # Try resolving by basename under known roots.
            for base in [PROCESSED_ROOT, PROCESSED_ROOT.parent, Path("/kaggle/input"), Path.cwd()]:
                if base.exists():
                    hits = list(base.rglob(p.name))
                    if hits:
                        return str(hits[0])
    return ""

metadata["audio_path"] = metadata.apply(infer_audio_path, axis=1)

if "dataset" not in metadata.columns:
    metadata["dataset"] = "unknown_dataset"
if "sample_id" not in metadata.columns:
    metadata["sample_id"] = metadata["dataset"].astype(str) + "_" + metadata.index.astype(str)

speaker_col = None
for c in ["speaker_global", "speaker_id", "speaker", "actor", "actor_id"]:
    if c in metadata.columns:
        speaker_col = c
        break
if speaker_col is None:
    # Last-resort speaker id prevents hard crash but strict split becomes sample-level strict.
    metadata["speaker_global"] = metadata["dataset"].astype(str) + "_" + metadata["sample_id"].astype(str)
    speaker_col = "speaker_global"

metadata = metadata[
    metadata["emotion"].isin(COMMON_EMOTIONS) &
    metadata["audio_path"].astype(str).str.len().gt(0)
].copy().reset_index(drop=True)

metadata["emotion"] = metadata["emotion"].astype(str)
metadata["label_id"] = metadata["emotion"].map(LABEL_TO_ID).astype(int)
metadata["dataset"] = metadata["dataset"].astype(str)
metadata["sample_id"] = metadata["sample_id"].astype(str)
metadata["speaker_for_split"] = metadata["dataset"].astype(str) + "::" + metadata[speaker_col].astype(str)

if QUICK_RUN:
    metadata = (
        metadata.groupby(["dataset", "emotion"], group_keys=False)
        .apply(lambda x: x.sample(min(len(x), max(1, QUICK_RUN_SAMPLES_PER_DATASET // NUM_CLASSES)), random_state=SEED))
        .reset_index(drop=True)
    )
    metadata["label_id"] = metadata["emotion"].map(LABEL_TO_ID).astype(int)

DOMAIN_NAMES = sorted(metadata["dataset"].unique())
DOMAIN_TO_ID = {d: i for i, d in enumerate(DOMAIN_NAMES)}
metadata["domain_id"] = metadata["dataset"].map(DOMAIN_TO_ID).astype(int)
NUM_DOMAINS = len(DOMAIN_NAMES)

print("PROCESSED_ROOT:", PROCESSED_ROOT)
print("AUDIO_DIR:", AUDIO_DIR)
print("metadata:", metadata.shape)
print("datasets:", DOMAIN_NAMES)
print("speakers:", metadata["speaker_for_split"].nunique())
display(pd.crosstab(metadata["dataset"], metadata["emotion"]).reindex(columns=COMMON_EMOTIONS))
metadata.to_csv(REPORT_DIR / "metadata_used_for_comparison.csv", index=False)

## 3. Shared Split Protocols

Tất cả route dùng cùng `protocols`.

- Random protocol: 60/20/20 theo sample.
- Strict protocol: group split theo `speaker_for_split`, không overlap speaker.
- Single dataset protocol: sinh thêm random/strict cho từng dataset.

In [ ]:
def can_stratify(labels, min_each=2):
    vals, counts = np.unique(labels, return_counts=True)
    return len(vals) >= 2 and np.all(counts >= min_each)

def make_random_60_20_20_split(df, source_indices=None):
    if source_indices is None:
        source_indices = np.arange(len(df))
    source_indices = np.asarray(source_indices)
    sub = df.iloc[source_indices].reset_index(drop=False)
    local_idx = np.arange(len(sub))
    strat = sub["label_id"].values if can_stratify(sub["label_id"].values, 2) else None
    train_local, temp_local = train_test_split(
        local_idx, test_size=0.40, random_state=SEED, stratify=strat
    )
    temp_labels = sub.iloc[temp_local]["label_id"].values
    strat_temp = temp_labels if can_stratify(temp_labels, 2) else None
    val_local, test_local = train_test_split(
        temp_local, test_size=0.50, random_state=SEED, stratify=strat_temp
    )
    return {
        "train": sub.iloc[train_local]["index"].values,
        "validation": sub.iloc[val_local]["index"].values,
        "test": sub.iloc[test_local]["index"].values,
    }

def make_strict_speaker_split(df, source_indices=None):
    if source_indices is None:
        source_indices = np.arange(len(df))
    source_indices = np.asarray(source_indices)
    sub = df.iloc[source_indices].reset_index(drop=False)
    if sub["speaker_for_split"].nunique() < 3:
        raise ValueError("Need at least 3 speakers for strict speaker split")
    local_idx = np.arange(len(sub))
    groups = sub["speaker_for_split"].values
    outer = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
    train_val_local, test_local = next(outer.split(local_idx, groups=groups))
    train_val_sub = sub.iloc[train_val_local]
    if train_val_sub["speaker_for_split"].nunique() < 2:
        raise ValueError("Not enough speakers for train/validation after test split")
    inner = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=SEED)
    train_rel, val_rel = next(inner.split(
        np.arange(len(train_val_sub)),
        groups=train_val_sub["speaker_for_split"].values
    ))
    train_local = train_val_local[train_rel]
    val_local = train_val_local[val_rel]
    return {
        "train": sub.iloc[train_local]["index"].values,
        "validation": sub.iloc[val_local]["index"].values,
        "test": sub.iloc[test_local]["index"].values,
    }

protocols = {}
if "combined_random_60_20_20_all4" in RUN_PROTOCOLS:
    protocols["combined_random_60_20_20_all4"] = make_random_60_20_20_split(metadata)
if "combined_strict_speaker_all4" in RUN_PROTOCOLS:
    protocols["combined_strict_speaker_all4"] = make_strict_speaker_split(metadata)

if RUN_SINGLE_DATASET_PROTOCOLS:
    for dataset_name in sorted(metadata["dataset"].unique()):
        idx = metadata.index[metadata["dataset"].eq(dataset_name)].values
        safe = str(dataset_name).lower().replace("-", "_").replace(" ", "_").replace("/", "_")
        if len(idx) < max(30, NUM_CLASSES * 3):
            print("Skip small dataset:", dataset_name, len(idx))
            continue
        try:
            protocols[f"single_random_60_20_20_{safe}"] = make_random_60_20_20_split(metadata, idx)
        except Exception as exc:
            print(f"Skip single random for {dataset_name}:", exc)
        try:
            protocols[f"single_strict_speaker_{safe}"] = make_strict_speaker_split(metadata, idx)
        except Exception as exc:
            print(f"Skip single strict for {dataset_name}:", exc)

# Save split manifest.
split_manifest_rows = []
for protocol_name, split in protocols.items():
    for split_name, indices in split.items():
        for idx in indices:
            split_manifest_rows.append({
                "protocol": protocol_name,
                "split": split_name,
                "row_index": int(idx),
                "sample_id": metadata.iloc[idx]["sample_id"],
                "dataset": metadata.iloc[idx]["dataset"],
                "emotion": metadata.iloc[idx]["emotion"],
                "speaker_for_split": metadata.iloc[idx]["speaker_for_split"],
                "audio_path": metadata.iloc[idx]["audio_path"],
            })
split_manifest_df = pd.DataFrame(split_manifest_rows)
split_manifest_path = SPLIT_DIR / "shared_split_manifest.csv"
split_manifest_df.to_csv(split_manifest_path, index=False)

for name, split in protocols.items():
    print("=" * 100)
    print(name, {k: len(v) for k, v in split.items()})
    for split_name, indices in split.items():
        print(split_name)
        display(pd.crosstab(metadata.iloc[indices]["dataset"], metadata.iloc[indices]["emotion"]).reindex(columns=COMMON_EMOTIONS))
    if "strict" in name:
        sets = {k: set(metadata.iloc[v]["speaker_for_split"]) for k, v in split.items()}
        print("speaker overlap train/validation:", len(sets["train"] & sets["validation"]))
        print("speaker overlap train/test:", len(sets["train"] & sets["test"]))
        print("speaker overlap validation/test:", len(sets["validation"] & sets["test"]))

print("Saved split manifest:", split_manifest_path)

## 4. Shared Utilities

In [ ]:
def crop_or_pad(y, target_len, mode="center"):
    y = np.asarray(y, dtype=np.float32)
    if len(y) > target_len:
        if mode == "random":
            start = np.random.randint(0, len(y) - target_len + 1)
        elif mode == "left":
            start = 0
        else:
            start = max(0, (len(y) - target_len) // 2)
        return y[start:start + target_len]
    if len(y) < target_len:
        return np.pad(y, (0, target_len - len(y)))
    return y

def load_fixed_audio(path, sr, duration, offset=0.0, center_crop=True):
    y, _ = librosa.load(path, sr=sr, mono=True, duration=duration, offset=offset)
    y = y.astype(np.float32)
    if len(y):
        y = y - np.mean(y)
    peak = np.max(np.abs(y)) if len(y) else 0.0
    if peak > 1.0:
        y = y / peak
    return crop_or_pad(y, int(sr * duration), mode="center" if center_crop else "left")

def trim_silence(y, top_db=30):
    if len(y) == 0:
        return y
    try:
        yt, _ = librosa.effects.trim(y, top_db=top_db)
        if len(yt) >= int(0.25 * TARGET_SR):
            return yt.astype(np.float32)
    except Exception:
        pass
    return y.astype(np.float32)

def rms_normalize(y, target_dbfs=-25.0):
    y = np.asarray(y, dtype=np.float32)
    rms = float(np.sqrt(np.mean(y ** 2) + 1e-12))
    target = 10 ** (target_dbfs / 20.0)
    if rms > 1e-8:
        y = y * (target / rms)
    return np.clip(y, -1.0, 1.0).astype(np.float32)

def resize_time_matrix(mat, target_frames):
    mat = np.asarray(mat, dtype=np.float32)
    if mat.ndim == 1:
        mat = mat[None, :]
    if mat.shape[1] == target_frames:
        return mat.astype(np.float32)
    if mat.shape[1] <= 1:
        return np.repeat(mat, target_frames, axis=1).astype(np.float32)
    old = np.linspace(0, 1, mat.shape[1])
    new = np.linspace(0, 1, target_frames)
    return np.stack([np.interp(new, old, row) for row in mat], axis=0).astype(np.float32)

def pad_vector(vec, target_len):
    vec = np.asarray(vec, dtype=np.float32)
    if len(vec) > target_len:
        return vec[:target_len]
    if len(vec) < target_len:
        return np.pad(vec, (0, target_len - len(vec)))
    return vec

def add_noise(y, scale_low=0.001, scale_high=0.010):
    level = np.random.uniform(scale_low, scale_high)
    return (y + level * np.random.randn(len(y))).astype(np.float32)

def add_noise_relative(y):
    noise_amp = 0.035 * np.random.uniform() * max(np.max(np.abs(y)), 1e-6)
    return (y + noise_amp * np.random.normal(size=len(y))).astype(np.float32)

def pitch_shift(y, sr, n_steps=None):
    if n_steps is None:
        n_steps = np.random.uniform(-1.5, 1.5)
    try:
        return librosa.effects.pitch_shift(y, sr=sr, n_steps=n_steps).astype(np.float32)
    except Exception:
        return y.astype(np.float32)

def time_stretch_fixed(y, rate=None):
    if rate is None:
        rate = np.random.uniform(0.95, 1.05)
    try:
        out = librosa.effects.time_stretch(y, rate=rate)
    except Exception:
        return y.astype(np.float32)
    return crop_or_pad(out, len(y), mode="center")

def compute_metrics(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "uar": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "macro_precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "macro_recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
    }

def softmax_np(logits):
    logits = np.asarray(logits, dtype=np.float32)
    logits = logits - logits.max(axis=1, keepdims=True)
    ex = np.exp(logits)
    return ex / np.maximum(ex.sum(axis=1, keepdims=True), 1e-12)

def metrics_row(protocol, route, model_name, split_name, result, n_samples, extra=None):
    row = {
        "protocol": protocol,
        "route": route,
        "model": model_name,
        "split": split_name,
        "n_samples": int(n_samples),
        **{k: float(v) for k, v in result.items() if isinstance(v, (int, float, np.floating))},
    }
    if extra:
        row.update(extra)
    return row

def save_prediction_csv(protocol, route, model_name, split_name, indices, probs, y_true):
    pred = probs.argmax(axis=1)
    out = metadata.iloc[indices][["sample_id", "dataset", "emotion", "speaker_for_split", "audio_path"]].copy()
    out.insert(0, "row_index", indices)
    out["true"] = [ID_TO_LABEL[int(x)] for x in y_true]
    out["pred"] = [ID_TO_LABEL[int(x)] for x in pred]
    out["confidence"] = probs.max(axis=1)
    for i, e in ID_TO_LABEL.items():
        out[f"p_{e}"] = probs[:, i]
    fname = f"predictions_{protocol}_{route}_{model_name}_{split_name}.csv".replace("/", "_")
    path = PRED_DIR / fname
    out.to_csv(path, index=False)
    return path

def class_weights_from_indices(indices):
    labels = metadata.iloc[indices]["label_id"].values
    counts = np.bincount(labels, minlength=NUM_CLASSES)
    weights = counts.sum() / np.maximum(counts, 1)
    weights = weights / weights.mean()
    return torch.tensor(weights, dtype=torch.float32, device=DEVICE)

def make_weighted_sampler(indices):
    labels = metadata.iloc[indices]["label_id"].values
    counts = np.bincount(labels, minlength=NUM_CLASSES)
    sample_weights = np.asarray([1.0 / max(counts[y], 1) for y in labels], dtype=np.float32)
    return WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

def move_to_device(batch):
    out = {}
    for k, v in batch.items():
        out[k] = v.to(DEVICE, non_blocking=True) if torch.is_tensor(v) else v
    return out

GLOBAL_METRICS = []
GLOBAL_HISTORY = []
PREDICTION_REGISTRY = defaultdict(dict)

<a id="route-1-emonity-style"></a>
# Route 1. Emonity-Style Full Pipeline

Pipeline:

```text
metadata/audio
  -> load audio
  -> Emonity-like augmentation only on train
  -> features for 1D: MFCC + delta + delta-delta + spectral + chroma + ZCR + RMS
  -> features for 2D: log-Mel image
  -> train 1D-CNN, 2D-CNN, CNN-BiLSTM
  -> validation macro-F1 weighted ensemble
```

In [ ]:
# =========================
# Route 1: Emonity Features
# =========================

def emonity_augment(y, sr):
    out = y.copy()
    if np.random.rand() < 0.50:
        out = add_noise(out, 0.001, 0.010)
    if np.random.rand() < 0.50:
        out = pitch_shift(out, sr)
    if np.random.rand() < 0.50:
        out = time_stretch_fixed(out)
    return out.astype(np.float32)

def extract_emonity_features_from_audio(y, sr, target_frames=EMONITY_TARGET_FRAMES):
    n_fft = min(2048, max(256, len(y) // 4))
    hop = min(512, max(128, len(y) // 8))

    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=20, n_fft=n_fft, hop_length=hop)
    if mfcc.shape[1] >= 9:
        mfcc = np.concatenate([
            mfcc,
            librosa.feature.delta(mfcc, width=5),
            librosa.feature.delta(mfcc, order=2, width=5),
        ], axis=0)

    mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=64, n_fft=n_fft, hop_length=hop)
    logmel = librosa.power_to_db(mel, ref=np.max)

    centroid = librosa.feature.spectral_centroid(y=y, sr=sr, n_fft=n_fft, hop_length=hop)
    bandwidth = librosa.feature.spectral_bandwidth(y=y, sr=sr, n_fft=n_fft, hop_length=hop)
    rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr, n_fft=n_fft, hop_length=hop)
    zcr = librosa.feature.zero_crossing_rate(y, hop_length=hop)
    rms = librosa.feature.rms(y=y, hop_length=hop)
    chroma = librosa.feature.chroma_stft(y=y, sr=sr, n_fft=n_fft, hop_length=hop)

    x1d = np.concatenate([
        resize_time_matrix(mfcc, target_frames),
        resize_time_matrix(centroid, target_frames),
        resize_time_matrix(bandwidth, target_frames),
        resize_time_matrix(rolloff, target_frames),
        resize_time_matrix(chroma, target_frames),
        resize_time_matrix(zcr, target_frames),
        resize_time_matrix(rms, target_frames),
    ], axis=0).T.astype(np.float32)

    x2d = resize_time_matrix(logmel, target_frames).astype(np.float32)
    x2d = np.stack([x2d, x2d, x2d], axis=0)
    return x1d, x2d

def build_emonity_base_cache():
    cache = CACHE_DIR / f"route1_emonity_base_{len(metadata)}_{EMONITY_SR}_{EMONITY_DURATION:g}s_{EMONITY_TARGET_FRAMES}f.npz"
    if cache.exists():
        data = np.load(cache, allow_pickle=True)
        return data["x1d"], data["x2d"]
    x1d_list, x2d_list = [], []
    for row in tqdm(metadata.itertuples(index=False), total=len(metadata), desc="Route 1 Emonity base features"):
        y = load_fixed_audio(row.audio_path, sr=EMONITY_SR, duration=EMONITY_DURATION, offset=0.0)
        x1d, x2d = extract_emonity_features_from_audio(y, EMONITY_SR)
        x1d_list.append(x1d)
        x2d_list.append(x2d)
    x1d = np.stack(x1d_list).astype(np.float32)
    x2d = np.stack(x2d_list).astype(np.float32)
    np.savez_compressed(cache, x1d=x1d, x2d=x2d)
    print("Saved:", cache, x1d.shape, x2d.shape)
    return x1d, x2d

class EmonityDataset(Dataset):
    def __init__(self, x1d, x2d, y, indices=None, return_indices=None):
        self.x1d = x1d
        self.x2d = x2d
        self.y = y.astype(np.int64)
        self.indices = np.arange(len(y)) if indices is None else np.asarray(indices)
        self.return_indices = np.asarray(return_indices) if return_indices is not None else self.indices

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        j = self.indices[i]
        return {
            "x1d": torch.tensor(self.x1d[j], dtype=torch.float32),
            "x2d": torch.tensor(self.x2d[j], dtype=torch.float32),
            "label": torch.tensor(self.y[j], dtype=torch.long),
            "row_index": torch.tensor(int(self.return_indices[i]), dtype=torch.long),
        }

def make_emonity_loaders(split):
    base_x1d, base_x2d = build_emonity_base_cache()
    y_all = metadata["label_id"].values.astype(np.int64)

    train_x1d = [base_x1d[split["train"]]]
    train_x2d = [base_x2d[split["train"]]]
    train_y = [y_all[split["train"]]]
    train_row_indices = [split["train"]]

    if EMONITY_AUGMENT_TRAIN:
        for copy_id in range(EMONITY_AUG_COPIES):
            aug_x1d, aug_x2d, aug_y, aug_rows = [], [], [], []
            for idx in tqdm(split["train"], desc=f"Route 1 train augmentation copy {copy_id+1}", leave=False):
                row = metadata.iloc[int(idx)]
                y = load_fixed_audio(row.audio_path, sr=EMONITY_SR, duration=EMONITY_DURATION, offset=0.0)
                y = emonity_augment(y, EMONITY_SR)
                a1, a2 = extract_emonity_features_from_audio(y, EMONITY_SR)
                aug_x1d.append(a1)
                aug_x2d.append(a2)
                aug_y.append(int(row.label_id))
                aug_rows.append(int(idx))
            train_x1d.append(np.stack(aug_x1d).astype(np.float32))
            train_x2d.append(np.stack(aug_x2d).astype(np.float32))
            train_y.append(np.asarray(aug_y, dtype=np.int64))
            train_row_indices.append(np.asarray(aug_rows, dtype=np.int64))

    x1d_train = np.concatenate(train_x1d, axis=0)
    x2d_train = np.concatenate(train_x2d, axis=0)
    y_train = np.concatenate(train_y, axis=0)
    row_train = np.concatenate(train_row_indices, axis=0)

    # Scale 1D features using train only.
    scaler = StandardScaler().fit(x1d_train.reshape(-1, x1d_train.shape[-1]))
    x1d_train = scaler.transform(x1d_train.reshape(-1, x1d_train.shape[-1])).reshape(x1d_train.shape).astype(np.float32)
    x1d_scaled_all = scaler.transform(base_x1d.reshape(-1, base_x1d.shape[-1])).reshape(base_x1d.shape).astype(np.float32)

    # Normalize 2D log-mel using train only.
    mean = x2d_train.mean(axis=(0, 2, 3), keepdims=True)
    std = x2d_train.std(axis=(0, 2, 3), keepdims=True) + 1e-6
    x2d_train = ((x2d_train - mean) / std).astype(np.float32)
    x2d_scaled_all = ((base_x2d - mean) / std).astype(np.float32)

    train_ds = EmonityDataset(x1d_train, x2d_train, y_train, return_indices=row_train)
    val_ds = EmonityDataset(x1d_scaled_all, x2d_scaled_all, y_all, indices=split["validation"], return_indices=split["validation"])
    test_ds = EmonityDataset(x1d_scaled_all, x2d_scaled_all, y_all, indices=split["test"], return_indices=split["test"])

    return {
        "train": DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS),
        "validation": DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS),
        "test": DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS),
    }

class Emonity1DCNN(nn.Module):
    def __init__(self, in_dim, num_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(in_dim, 128, kernel_size=5, padding=2), nn.BatchNorm1d(128), nn.ReLU(), nn.MaxPool1d(2), nn.Dropout(0.25),
            nn.Conv1d(128, 192, kernel_size=5, padding=2), nn.BatchNorm1d(192), nn.ReLU(), nn.MaxPool1d(2), nn.Dropout(0.30),
            nn.Conv1d(192, 256, kernel_size=3, padding=1), nn.BatchNorm1d(256), nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
        )
        self.head = nn.Sequential(nn.Flatten(), nn.Linear(256, 128), nn.ReLU(), nn.Dropout(0.35), nn.Linear(128, num_classes))

    def forward(self, batch):
        x = batch["x1d"].transpose(1, 2)
        return self.head(self.net(x))

class Emonity2DCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2), nn.Dropout2d(0.15),
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2), nn.Dropout2d(0.20),
            nn.Conv2d(64, 128, kernel_size=3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.head = nn.Sequential(nn.Flatten(), nn.Linear(128, 128), nn.ReLU(), nn.Dropout(0.35), nn.Linear(128, num_classes))

    def forward(self, batch):
        return self.head(self.net(batch["x2d"]))

class EmonityCNNBiLSTM(nn.Module):
    def __init__(self, in_dim, num_classes):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(in_dim, 128, kernel_size=5, padding=2), nn.BatchNorm1d(128), nn.ReLU(), nn.MaxPool1d(2), nn.Dropout(0.20),
            nn.Conv1d(128, 192, kernel_size=3, padding=1), nn.BatchNorm1d(192), nn.ReLU(), nn.MaxPool1d(2), nn.Dropout(0.25),
        )
        self.rnn = nn.LSTM(192, 128, num_layers=1, batch_first=True, bidirectional=True)
        self.attn = nn.Sequential(nn.Linear(256, 128), nn.Tanh(), nn.Linear(128, 1))
        self.head = nn.Sequential(nn.Linear(256, 128), nn.ReLU(), nn.Dropout(0.35), nn.Linear(128, num_classes))

    def forward(self, batch):
        x = batch["x1d"].transpose(1, 2)
        x = self.conv(x).transpose(1, 2)
        h, _ = self.rnn(x)
        w = torch.softmax(self.attn(h), dim=1)
        pooled = (h * w).sum(dim=1)
        return self.head(pooled)

def build_route1_model(model_name, sample_batch):
    if model_name == "emonity_1d_cnn":
        return Emonity1DCNN(sample_batch["x1d"].shape[-1], NUM_CLASSES)
    if model_name == "emonity_2d_cnn":
        return Emonity2DCNN(NUM_CLASSES)
    if model_name == "emonity_cnn_bilstm":
        return EmonityCNNBiLSTM(sample_batch["x1d"].shape[-1], NUM_CLASSES)
    raise ValueError(model_name)

In [ ]:
# ======================
# Route 1 Training Cell
# ======================

ROUTE1_MODELS = [m.strip() for m in os.getenv(
    "ROUTE1_MODELS",
    "emonity_1d_cnn,emonity_2d_cnn,emonity_cnn_bilstm"
).split(",") if m.strip()]

def predict_generic(model, loader):
    model.eval()
    logits_list, y_list, idx_list = [], [], []
    with torch.no_grad():
        for batch in loader:
            batch = move_to_device(batch)
            logits = model(batch)
            logits_list.append(logits.detach().cpu().numpy())
            y_list.append(batch["label"].detach().cpu().numpy())
            idx_list.append(batch["row_index"].detach().cpu().numpy())
    logits = np.concatenate(logits_list, axis=0)
    y = np.concatenate(y_list, axis=0)
    idx = np.concatenate(idx_list, axis=0)
    return logits, y, idx

def train_generic_model(protocol, route, model_name, model, loaders, class_weights=None):
    model = model.to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    best_state, best_val, best_epoch = None, -1.0, 0
    patience_left = PATIENCE
    history = []
    start = time.time()

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        train_loss, n_train = 0.0, 0
        train_pred, train_true = [], []
        for batch in loaders["train"]:
            batch = move_to_device(batch)
            logits = model(batch)
            loss = criterion(logits, batch["label"])
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
            train_loss += float(loss.item()) * batch["label"].size(0)
            n_train += batch["label"].size(0)
            train_pred.extend(logits.argmax(dim=1).detach().cpu().numpy().tolist())
            train_true.extend(batch["label"].detach().cpu().numpy().tolist())

        val_logits, val_y, _ = predict_generic(model, loaders["validation"])
        val_pred = val_logits.argmax(axis=1)
        train_m = compute_metrics(np.asarray(train_true), np.asarray(train_pred))
        val_m = compute_metrics(val_y, val_pred)
        row = {
            "protocol": protocol, "route": route, "model": model_name, "epoch": epoch,
            "train_loss": train_loss / max(n_train, 1),
            **{f"train_{k}": v for k, v in train_m.items()},
            **{f"val_{k}": v for k, v in val_m.items()},
        }
        history.append(row)
        print(f"{protocol} | {route} | {model_name} | epoch {epoch:03d} | val_macro_f1={val_m['macro_f1']:.4f}")

        if val_m["macro_f1"] > best_val:
            best_val = val_m["macro_f1"]
            best_epoch = epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            patience_left = PATIENCE
        else:
            patience_left -= 1
            if patience_left <= 0:
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    split_results, pred_paths = {}, {}
    for split_name in ["train", "validation", "test"]:
        logits, y, idx = predict_generic(model, loaders[split_name])
        probs = softmax_np(logits)
        pred = probs.argmax(axis=1)
        split_results[split_name] = compute_metrics(y, pred)
        pred_paths[split_name] = save_prediction_csv(protocol, route, model_name, split_name, idx, probs, y)

    model_path = MODEL_DIR / f"{protocol}_{route}_{model_name}.pt".replace("/", "_")
    torch.save({"model_state_dict": model.state_dict(), "model": model_name, "protocol": protocol, "route": route}, model_path)
    train_time = time.time() - start

    for h in history:
        GLOBAL_HISTORY.append(h)
    for split_name, result in split_results.items():
        GLOBAL_METRICS.append(metrics_row(
            protocol, route, model_name, split_name, result,
            n_samples=len(loaders[split_name].dataset),
            extra={"train_time_sec": train_time, "best_epoch": best_epoch, "best_val_macro_f1": best_val}
        ))
    PREDICTION_REGISTRY[(protocol, route)][model_name] = pred_paths
    return split_results, pred_paths

def val_weighted_ensemble(protocol, route, members, ensemble_name):
    key = (protocol, route)
    if not all(m in PREDICTION_REGISTRY[key] for m in members):
        print("Skip ensemble, missing members:", protocol, route, members)
        return

    weights = []
    for m in members:
        val_df = pd.read_csv(PREDICTION_REGISTRY[key][m]["validation"])
        prob_cols = [f"p_{e}" for e in COMMON_EMOTIONS]
        y = val_df["true"].map(LABEL_TO_ID).values
        p = val_df[prob_cols].values
        weights.append(max(0.0, f1_score(y, p.argmax(axis=1), average="macro", zero_division=0)))

    weights = np.asarray(weights, dtype=np.float32)
    if weights.sum() <= 0:
        weights = np.ones_like(weights) / len(weights)
    else:
        # Selective: very weak members can become zero.
        threshold = float(os.getenv("ENSEMBLE_MIN_RELATIVE_WEIGHT", "0.80"))
        max_w = weights.max()
        weights = np.where(weights >= threshold * max_w, weights, 0.0)
        if weights.sum() <= 0:
            weights = np.ones(len(members), dtype=np.float32)
        weights = weights / weights.sum()

    print("Ensemble weights:", protocol, route, dict(zip(members, weights.round(4))))
    weight_rows = [{"protocol": protocol, "route": route, "ensemble": ensemble_name, "member": m, "weight": float(w)}
                   for m, w in zip(members, weights)]
    pd.DataFrame(weight_rows).to_csv(REPORT_DIR / f"weights_{protocol}_{route}_{ensemble_name}.csv", index=False)

    for split_name in ["validation", "test"]:
        base_df = pd.read_csv(PREDICTION_REGISTRY[key][members[0]][split_name])
        prob_cols = [f"p_{e}" for e in COMMON_EMOTIONS]
        probs = np.zeros((len(base_df), NUM_CLASSES), dtype=np.float32)
        for w, m in zip(weights, members):
            df = pd.read_csv(PREDICTION_REGISTRY[key][m][split_name])
            probs += float(w) * df[prob_cols].values.astype(np.float32)
        y = base_df["true"].map(LABEL_TO_ID).values
        pred = probs.argmax(axis=1)
        result = compute_metrics(y, pred)
        idx = base_df["row_index"].values
        path = save_prediction_csv(protocol, route, ensemble_name, split_name, idx, probs, y)
        PREDICTION_REGISTRY[key][ensemble_name] = {**PREDICTION_REGISTRY[key].get(ensemble_name, {}), split_name: path}
        GLOBAL_METRICS.append(metrics_row(protocol, route, ensemble_name, split_name, result, len(y), extra={"ensemble_members": ",".join(members)}))

if "1" in RUN_ROUTES:
    for protocol_name, split in protocols.items():
        print("=" * 110)
        print("ROUTE 1 EMONITY STYLE |", protocol_name)
        loaders = make_emonity_loaders(split)
        sample_batch = next(iter(loaders["train"]))
        class_weights = class_weights_from_indices(split["train"])
        for model_name in ROUTE1_MODELS:
            model = build_route1_model(model_name, sample_batch)
            train_generic_model(protocol_name, "route1_emonity", model_name, model, loaders, class_weights)
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        val_weighted_ensemble(protocol_name, "route1_emonity", ROUTE1_MODELS, "emonity_val_weighted_ensemble")
else:
    print("Skip Route 1 because RUN_ROUTES does not contain '1'.")

<a id="route-2-cnn-lstm-clstm-style"></a>
# Route 2. CNN/LSTM/CLSTM-Style Full Pipeline

Pipeline:

```text
metadata/audio
  -> librosa.load(duration=2.5, offset=0.6, sr=22050)
  -> feature = flatten([ZCR, RMS/RMSE, MFCC])
  -> train augmentation only:
       original
       noise
       pitch
       pitch+noise
  -> CNN / LSTM / CNN-LSTM
  -> validation macro-F1 weighted ensemble
```

In [ ]:
# ==================================
# Route 2: CNN/LSTM/CLSTM Features
# ==================================

def extract_clstm_flat_from_audio(y, sr=CLSTM_SR, frame_length=2048, hop_length=512):
    zcr = librosa.feature.zero_crossing_rate(y, frame_length=frame_length, hop_length=hop_length).squeeze()
    rms = librosa.feature.rms(y=y, frame_length=frame_length, hop_length=hop_length).squeeze()
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=20, n_fft=frame_length, hop_length=hop_length).T.reshape(-1)
    return np.hstack([np.ravel(zcr), np.ravel(rms), mfcc]).astype(np.float32)

def build_clstm_base_cache():
    cache = CACHE_DIR / f"route2_clstm_base_{len(metadata)}_{CLSTM_SR}_{CLSTM_DURATION:g}s_offset{CLSTM_OFFSET:g}.npz"
    if cache.exists():
        data = np.load(cache, allow_pickle=True)
        return data["flat"], int(data["target_len"])
    vecs = []
    for row in tqdm(metadata.itertuples(index=False), total=len(metadata), desc="Route 2 CLSTM base features"):
        y = load_fixed_audio(row.audio_path, sr=CLSTM_SR, duration=CLSTM_DURATION, offset=CLSTM_OFFSET, center_crop=False)
        vecs.append(extract_clstm_flat_from_audio(y, CLSTM_SR))
    target_len = max(len(v) for v in vecs)
    flat = np.stack([pad_vector(v, target_len) for v in vecs]).astype(np.float32)
    np.savez_compressed(cache, flat=flat, target_len=np.asarray(target_len))
    print("Saved:", cache, flat.shape)
    return flat, target_len

class CLSTMDataset(Dataset):
    def __init__(self, flat, y, indices=None, return_indices=None):
        self.flat = flat.astype(np.float32)
        self.y = y.astype(np.int64)
        self.indices = np.arange(len(y)) if indices is None else np.asarray(indices)
        self.return_indices = np.asarray(return_indices) if return_indices is not None else self.indices

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        j = self.indices[i]
        x = self.flat[j]
        return {
            "flat": torch.tensor(x, dtype=torch.float32),
            "seq": torch.tensor(x[:, None], dtype=torch.float32),
            "label": torch.tensor(self.y[j], dtype=torch.long),
            "row_index": torch.tensor(int(self.return_indices[i]), dtype=torch.long),
        }

def make_clstm_loaders(split):
    base_flat, target_len = build_clstm_base_cache()
    y_all = metadata["label_id"].values.astype(np.int64)

    train_flat = [base_flat[split["train"]]]
    train_y = [y_all[split["train"]]]
    train_rows = [split["train"]]

    if CLSTM_AUGMENT_TRAIN:
        aug_vecs, aug_y, aug_rows = [], [], []
        for idx in tqdm(split["train"], desc="Route 2 train augmentation", leave=False):
            row = metadata.iloc[int(idx)]
            y = load_fixed_audio(row.audio_path, sr=CLSTM_SR, duration=CLSTM_DURATION, offset=CLSTM_OFFSET, center_crop=False)
            variants = [
                add_noise_relative(y),
                pitch_shift(y, CLSTM_SR, n_steps=0.7),
                add_noise_relative(pitch_shift(y, CLSTM_SR, n_steps=0.7)),
            ]
            for yy in variants:
                aug_vecs.append(pad_vector(extract_clstm_flat_from_audio(yy, CLSTM_SR), target_len))
                aug_y.append(int(row.label_id))
                aug_rows.append(int(idx))
        if aug_vecs:
            train_flat.append(np.stack(aug_vecs).astype(np.float32))
            train_y.append(np.asarray(aug_y, dtype=np.int64))
            train_rows.append(np.asarray(aug_rows, dtype=np.int64))

    flat_train = np.concatenate(train_flat, axis=0)
    y_train = np.concatenate(train_y, axis=0)
    row_train = np.concatenate(train_rows, axis=0)

    scaler = StandardScaler().fit(flat_train)
    flat_train = scaler.transform(flat_train).astype(np.float32)
    flat_scaled_all = scaler.transform(base_flat).astype(np.float32)

    train_ds = CLSTMDataset(flat_train, y_train, return_indices=row_train)
    val_ds = CLSTMDataset(flat_scaled_all, y_all, indices=split["validation"], return_indices=split["validation"])
    test_ds = CLSTMDataset(flat_scaled_all, y_all, indices=split["test"], return_indices=split["test"])

    return {
        "train": DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS),
        "validation": DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS),
        "test": DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS),
    }

class CLSTMBigCNN(nn.Module):
    def __init__(self, input_len, num_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(1, 256, kernel_size=5, padding=2), nn.BatchNorm1d(256), nn.ReLU(), nn.MaxPool1d(4), nn.Dropout(0.30),
            nn.Conv1d(256, 256, kernel_size=5, padding=2), nn.BatchNorm1d(256), nn.ReLU(), nn.MaxPool1d(4), nn.Dropout(0.30),
            nn.Conv1d(256, 128, kernel_size=3, padding=1), nn.BatchNorm1d(128), nn.ReLU(), nn.AdaptiveAvgPool1d(1),
        )
        self.head = nn.Sequential(nn.Flatten(), nn.Linear(128, 128), nn.ReLU(), nn.Dropout(0.35), nn.Linear(128, num_classes))

    def forward(self, batch):
        x = batch["flat"].unsqueeze(1)
        return self.head(self.net(x))

class CLSTMLSTM(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.rnn = nn.LSTM(1, 96, num_layers=2, batch_first=True, dropout=0.20, bidirectional=True)
        self.head = nn.Sequential(nn.Linear(192, 128), nn.ReLU(), nn.Dropout(0.35), nn.Linear(128, num_classes))

    def forward(self, batch):
        h, _ = self.rnn(batch["seq"])
        pooled = h.mean(dim=1)
        return self.head(pooled)

class CLSTMCNNLSTM(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(1, 128, kernel_size=5, padding=2), nn.BatchNorm1d(128), nn.ReLU(), nn.MaxPool1d(4), nn.Dropout(0.25),
            nn.Conv1d(128, 128, kernel_size=3, padding=1), nn.BatchNorm1d(128), nn.ReLU(), nn.MaxPool1d(2), nn.Dropout(0.25),
        )
        self.rnn = nn.LSTM(128, 96, batch_first=True, bidirectional=True)
        self.head = nn.Sequential(nn.Linear(192, 128), nn.ReLU(), nn.Dropout(0.35), nn.Linear(128, num_classes))

    def forward(self, batch):
        x = batch["flat"].unsqueeze(1)
        x = self.conv(x).transpose(1, 2)
        h, _ = self.rnn(x)
        return self.head(h.mean(dim=1))

def build_route2_model(model_name, sample_batch):
    input_len = sample_batch["flat"].shape[-1]
    if model_name == "clstm_big_cnn":
        return CLSTMBigCNN(input_len, NUM_CLASSES)
    if model_name == "clstm_lstm":
        return CLSTMLSTM(NUM_CLASSES)
    if model_name == "clstm_cnn_lstm":
        return CLSTMCNNLSTM(NUM_CLASSES)
    raise ValueError(model_name)

In [ ]:
# ======================
# Route 2 Training Cell
# ======================

ROUTE2_MODELS = [m.strip() for m in os.getenv(
    "ROUTE2_MODELS",
    "clstm_big_cnn,clstm_lstm,clstm_cnn_lstm"
).split(",") if m.strip()]

if "2" in RUN_ROUTES:
    for protocol_name, split in protocols.items():
        print("=" * 110)
        print("ROUTE 2 CNN/LSTM/CLSTM STYLE |", protocol_name)
        loaders = make_clstm_loaders(split)
        sample_batch = next(iter(loaders["train"]))
        class_weights = class_weights_from_indices(split["train"])
        for model_name in ROUTE2_MODELS:
            model = build_route2_model(model_name, sample_batch)
            train_generic_model(protocol_name, "route2_cnn_lstm_clstm", model_name, model, loaders, class_weights)
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        val_weighted_ensemble(protocol_name, "route2_cnn_lstm_clstm", ROUTE2_MODELS, "clstm_val_weighted_ensemble")
else:
    print("Skip Route 2 because RUN_ROUTES does not contain '2'.")

<a id="route-3-our-06f-domain-aware-mixture-adapter"></a>
# Route 3. Our 06F Domain-Aware Mixture Adapter

Pipeline lấy ý tưởng từ file `06F_Domain_Aware_Mixture_Adapter_SER.ipynb`:

```text
audio 16k
  -> trim silence
  -> RMS normalize
  -> fixed 4s crop/pad
  -> temporal branch: MFCC + delta + delta-delta + prosody/spectral frame features -> CNN + BiLSTM + attention
  -> spectral branch: log-Mel + deltas -> 2D CNN + SE
  -> stats branch: handcrafted stats
  -> optional emotion2vec vector
  -> temporal-spectral co-attention
  -> domain-aware mixture adapter
  -> auxiliary heads + main head
  -> validation selective branch fusion
```

`emotion2vec` mặc định tắt để notebook vẫn chạy được khi Kaggle chưa bật Internet. Nếu đã có embedding/cache riêng, có thể mở rộng cell `build_ours_feature_cache`.

In [ ]:
# =========================
# Route 3: Our 06F Features
# =========================

OURS_N_FFT = int(os.getenv("OURS_N_FFT", "1024"))
OURS_HOP = int(os.getenv("OURS_HOP", "512"))
OURS_N_MFCC = int(os.getenv("OURS_N_MFCC", "40"))
OURS_N_MELS = int(os.getenv("OURS_N_MELS", "96"))
OURS_TARGET_FRAMES = int(os.getenv("OURS_TARGET_FRAMES", str(math.ceil(TARGET_LENGTH / OURS_HOP))))

def summarize_feature_matrix(mat):
    mat = np.asarray(mat, dtype=np.float32)
    return np.concatenate([
        mat.mean(axis=1),
        mat.std(axis=1),
        mat.min(axis=1),
        mat.max(axis=1),
        np.percentile(mat, 25, axis=1),
        np.percentile(mat, 75, axis=1),
    ]).astype(np.float32)

def slopes_feature_matrix(mat):
    mat = np.asarray(mat, dtype=np.float32)
    t = np.arange(mat.shape[1], dtype=np.float32)
    t = (t - t.mean()) / (t.std() + 1e-6)
    denom = np.sum(t ** 2) + 1e-6
    return ((mat - mat.mean(axis=1, keepdims=True)) @ t / denom).astype(np.float32)

def segment_stats(mat, segments=3):
    chunks = np.array_split(mat, segments, axis=1)
    vals = []
    for c in chunks:
        vals.extend([c.mean(axis=1), c.std(axis=1)])
    return np.concatenate(vals).astype(np.float32)

def load_ours_audio(path):
    y, _ = librosa.load(path, sr=TARGET_SR, mono=True)
    y = y.astype(np.float32)
    y = trim_silence(y, top_db=30)
    y = rms_normalize(y, target_dbfs=-25.0)
    y = crop_or_pad(y, TARGET_LENGTH, mode="center")
    return y

def extract_ours_features_one(path):
    y = load_ours_audio(path)

    mfcc = librosa.feature.mfcc(y=y, sr=TARGET_SR, n_mfcc=OURS_N_MFCC, n_fft=OURS_N_FFT, hop_length=OURS_HOP)
    d1 = librosa.feature.delta(mfcc)
    d2 = librosa.feature.delta(mfcc, order=2)

    rms = librosa.feature.rms(y=y, frame_length=OURS_N_FFT, hop_length=OURS_HOP)
    zcr = librosa.feature.zero_crossing_rate(y, frame_length=OURS_N_FFT, hop_length=OURS_HOP)
    centroid = librosa.feature.spectral_centroid(y=y, sr=TARGET_SR, n_fft=OURS_N_FFT, hop_length=OURS_HOP)
    bandwidth = librosa.feature.spectral_bandwidth(y=y, sr=TARGET_SR, n_fft=OURS_N_FFT, hop_length=OURS_HOP)
    rolloff = librosa.feature.spectral_rolloff(y=y, sr=TARGET_SR, n_fft=OURS_N_FFT, hop_length=OURS_HOP)
    flatness = librosa.feature.spectral_flatness(y=y, n_fft=OURS_N_FFT, hop_length=OURS_HOP)
    contrast = librosa.feature.spectral_contrast(y=y, sr=TARGET_SR, n_fft=OURS_N_FFT, hop_length=OURS_HOP)

    temporal = np.concatenate([
        resize_time_matrix(mfcc, OURS_TARGET_FRAMES),
        resize_time_matrix(d1, OURS_TARGET_FRAMES),
        resize_time_matrix(d2, OURS_TARGET_FRAMES),
        resize_time_matrix(rms, OURS_TARGET_FRAMES),
        resize_time_matrix(zcr, OURS_TARGET_FRAMES),
        resize_time_matrix(centroid, OURS_TARGET_FRAMES),
        resize_time_matrix(bandwidth, OURS_TARGET_FRAMES),
        resize_time_matrix(rolloff, OURS_TARGET_FRAMES),
        resize_time_matrix(flatness, OURS_TARGET_FRAMES),
        resize_time_matrix(contrast, OURS_TARGET_FRAMES),
    ], axis=0).T.astype(np.float32)

    mel = librosa.feature.melspectrogram(y=y, sr=TARGET_SR, n_mels=OURS_N_MELS, n_fft=OURS_N_FFT, hop_length=OURS_HOP)
    logmel = librosa.power_to_db(mel, ref=np.max)
    logmel = resize_time_matrix(logmel, OURS_TARGET_FRAMES)
    dm1 = resize_time_matrix(librosa.feature.delta(logmel), OURS_TARGET_FRAMES)
    dm2 = resize_time_matrix(librosa.feature.delta(logmel, order=2), OURS_TARGET_FRAMES)
    spectral = np.stack([logmel, dm1, dm2], axis=0).astype(np.float32)

    stats_source = np.concatenate([
        resize_time_matrix(mfcc, OURS_TARGET_FRAMES),
        resize_time_matrix(rms, OURS_TARGET_FRAMES),
        resize_time_matrix(zcr, OURS_TARGET_FRAMES),
        resize_time_matrix(centroid, OURS_TARGET_FRAMES),
        resize_time_matrix(bandwidth, OURS_TARGET_FRAMES),
        resize_time_matrix(rolloff, OURS_TARGET_FRAMES),
        resize_time_matrix(flatness, OURS_TARGET_FRAMES),
    ], axis=0)
    chroma = librosa.feature.chroma_stft(y=y, sr=TARGET_SR, n_fft=OURS_N_FFT, hop_length=OURS_HOP)
    chroma = resize_time_matrix(chroma, OURS_TARGET_FRAMES)
    stats = np.concatenate([
        summarize_feature_matrix(stats_source),
        slopes_feature_matrix(stats_source),
        segment_stats(stats_source, segments=3),
        summarize_feature_matrix(chroma),
    ]).astype(np.float32)

    # Placeholder optional emotion2vec. Replace this block with cached emotion2vec if available.
    e2v = np.zeros((OURS_E2V_DIM,), dtype=np.float32)
    return temporal, spectral, stats, e2v

def build_ours_feature_cache():
    tag = f"route3_ours06f_{len(metadata)}_{TARGET_SR}_{TARGET_DURATION:g}s_{OURS_TARGET_FRAMES}f_e2v{OURS_E2V_DIM}"
    cache = CACHE_DIR / f"{tag}.npz"
    if cache.exists():
        data = np.load(cache, allow_pickle=True)
        return data["temporal"], data["spectral"], data["stats"], data["e2v"]

    temporal_list, spectral_list, stats_list, e2v_list = [], [], [], []
    for row in tqdm(metadata.itertuples(index=False), total=len(metadata), desc="Route 3 ours features"):
        t, s, st, e = extract_ours_features_one(row.audio_path)
        temporal_list.append(t)
        spectral_list.append(s)
        stats_list.append(st)
        e2v_list.append(e)

    X_t = np.stack(temporal_list).astype(np.float32)
    X_s = np.stack(spectral_list).astype(np.float32)
    X_stats = np.stack(stats_list).astype(np.float32)
    X_e2v = np.stack(e2v_list).astype(np.float32)
    np.savez_compressed(cache, temporal=X_t, spectral=X_s, stats=X_stats, e2v=X_e2v)
    print("Saved:", cache, X_t.shape, X_s.shape, X_stats.shape, X_e2v.shape)
    return X_t, X_s, X_stats, X_e2v

class OursDataset(Dataset):
    def __init__(self, indices, arrays):
        self.indices = np.asarray(indices)
        self.arrays = arrays

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        idx = int(self.indices[i])
        return {
            "temporal": torch.tensor(self.arrays["temporal"][idx], dtype=torch.float32),
            "spectral": torch.tensor(self.arrays["spectral"][idx], dtype=torch.float32),
            "stats": torch.tensor(self.arrays["stats"][idx], dtype=torch.float32),
            "e2v": torch.tensor(self.arrays["e2v"][idx], dtype=torch.float32),
            "domain": torch.tensor(int(metadata.iloc[idx]["domain_id"]), dtype=torch.long),
            "label": torch.tensor(int(metadata.iloc[idx]["label_id"]), dtype=torch.long),
            "row_index": torch.tensor(idx, dtype=torch.long),
        }

def make_ours_loaders(split):
    X_t, X_s, X_stats, X_e2v = build_ours_feature_cache()
    train_idx = split["train"]

    t_mean = X_t[train_idx].mean(axis=(0, 1), keepdims=True)
    t_std = X_t[train_idx].std(axis=(0, 1), keepdims=True) + 1e-6
    s_mean = X_s[train_idx].mean(axis=(0, 2, 3), keepdims=True)
    s_std = X_s[train_idx].std(axis=(0, 2, 3), keepdims=True) + 1e-6
    stats_scaler = StandardScaler().fit(X_stats[train_idx])
    e2v_scaler = StandardScaler().fit(X_e2v[train_idx]) if X_e2v.shape[1] > 0 else None

    arrays = {
        "temporal": ((X_t - t_mean) / t_std).astype(np.float32),
        "spectral": ((X_s - s_mean) / s_std).astype(np.float32),
        "stats": stats_scaler.transform(X_stats).astype(np.float32),
        "e2v": e2v_scaler.transform(X_e2v).astype(np.float32) if e2v_scaler is not None else X_e2v.astype(np.float32),
    }

    sampler = make_weighted_sampler(train_idx) if OURS_USE_BALANCED_SAMPLER else None
    train_ds = OursDataset(train_idx, arrays)
    val_ds = OursDataset(split["validation"], arrays)
    test_ds = OursDataset(split["test"], arrays)

    return {
        "train": DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=(sampler is None), sampler=sampler, num_workers=NUM_WORKERS),
        "validation": DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS),
        "test": DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS),
    }

In [ ]:
# ========================
# Route 3: Our 06F Model
# ========================

class AttentionPooling(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.score = nn.Sequential(nn.Linear(dim, max(8, dim // 2)), nn.Tanh(), nn.Linear(max(8, dim // 2), 1))

    def forward(self, x):
        w = torch.softmax(self.score(x), dim=1)
        return (x * w).sum(dim=1), w

class SE2D(nn.Module):
    def __init__(self, channels, reduction=8):
        super().__init__()
        hidden = max(4, channels // reduction)
        self.net = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(channels, hidden, kernel_size=1), nn.ReLU(),
            nn.Conv2d(hidden, channels, kernel_size=1), nn.Sigmoid()
        )

    def forward(self, x):
        return x * self.net(x)

class ResidualSEBlock(nn.Module):
    def __init__(self, in_ch, out_ch, pool=True):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(),
            nn.Conv2d(out_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch),
            SE2D(out_ch),
        )
        self.skip = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()
        self.pool = nn.MaxPool2d(2) if pool else nn.Identity()

    def forward(self, x):
        x = F.relu(self.conv(x) + self.skip(x))
        return self.pool(x)

class TemporalBranch06F(nn.Module):
    def __init__(self, in_dim, hidden=128):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(in_dim, 128, 5, padding=2), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.15),
            nn.Conv1d(128, 160, 3, padding=1), nn.BatchNorm1d(160), nn.ReLU(), nn.Dropout(0.15),
        )
        self.rnn = nn.LSTM(160, hidden, batch_first=True, bidirectional=True)
        self.pool = AttentionPooling(hidden * 2)

    def forward(self, x):
        x = self.conv(x.transpose(1, 2)).transpose(1, 2)
        h, _ = self.rnn(x)
        z, _ = self.pool(h)
        return z

class SpectralBranch06F(nn.Module):
    def __init__(self, out_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            ResidualSEBlock(3, 32),
            ResidualSEBlock(32, 64),
            ResidualSEBlock(64, 128),
            ResidualSEBlock(128, 192, pool=False),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.proj = nn.Sequential(nn.Flatten(), nn.Linear(192, out_dim), nn.ReLU(), nn.Dropout(0.25))

    def forward(self, x):
        return self.proj(self.net(x))

class MLPBranch06F(nn.Module):
    def __init__(self, in_dim, out_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, max(64, min(256, in_dim // 2 if in_dim > 1 else 64))),
            nn.BatchNorm1d(max(64, min(256, in_dim // 2 if in_dim > 1 else 64))),
            nn.ReLU(), nn.Dropout(0.25),
            nn.Linear(max(64, min(256, in_dim // 2 if in_dim > 1 else 64)), out_dim),
            nn.ReLU(), nn.Dropout(0.20),
        )

    def forward(self, x):
        return self.net(x)

class TemporalSpectralCoAttention06F(nn.Module):
    def __init__(self, dim=256):
        super().__init__()
        self.t_to_s = nn.Sequential(nn.Linear(dim * 2, dim), nn.Tanh(), nn.Linear(dim, dim), nn.Sigmoid())
        self.s_to_t = nn.Sequential(nn.Linear(dim * 2, dim), nn.Tanh(), nn.Linear(dim, dim), nn.Sigmoid())

    def forward(self, z_t, z_s):
        pair = torch.cat([z_t, z_s], dim=1)
        gate_ts = self.t_to_s(pair)
        gate_st = self.s_to_t(pair)
        z_t_ctx = z_t * gate_st + z_s * (1.0 - gate_st)
        z_s_ctx = z_s * gate_ts + z_t * (1.0 - gate_ts)
        return z_t_ctx, z_s_ctx, torch.cat([gate_ts, gate_st], dim=1)

class DomainAdapter06F(nn.Module):
    def __init__(self, dim, num_domains):
        super().__init__()
        self.adapters = nn.ModuleList([
            nn.Sequential(nn.Linear(dim, dim), nn.ReLU(), nn.Dropout(0.10), nn.Linear(dim, dim))
            for _ in range(max(1, num_domains))
        ])
        self.gate = nn.Sequential(nn.Linear(dim, max(8, dim // 2)), nn.ReLU(), nn.Linear(max(8, dim // 2), max(1, num_domains)))

    def forward(self, z):
        weights = torch.softmax(self.gate(z), dim=1)
        adapted = torch.stack([adapter(z) for adapter in self.adapters], dim=1)
        mixed = (adapted * weights.unsqueeze(-1)).sum(dim=1)
        return z + mixed, weights

class Ours06FDomainAwareMixtureAdapterSER(nn.Module):
    def __init__(self, temporal_dim, stats_dim, e2v_dim, num_domains, num_classes):
        super().__init__()
        self.temporal = TemporalBranch06F(temporal_dim, hidden=128)
        self.spectral = SpectralBranch06F(out_dim=256)
        self.stats = MLPBranch06F(stats_dim, out_dim=128)
        self.e2v = MLPBranch06F(max(e2v_dim, 1), out_dim=128)

        self.coattn = TemporalSpectralCoAttention06F(dim=256)
        fusion_dim = 256 + 256 + 256 + 128 + 128
        self.pre_fusion = nn.Sequential(nn.Linear(fusion_dim, 384), nn.ReLU(), nn.Dropout(0.30))
        self.adapter = DomainAdapter06F(384, num_domains)
        self.head = nn.Linear(384, num_classes)

        self.temporal_head = nn.Linear(256, num_classes)
        self.spectral_head = nn.Linear(256, num_classes)
        self.stats_head = nn.Linear(128, num_classes)
        self.e2v_head = nn.Linear(128, num_classes)
        self.domain_head = nn.Linear(384, num_domains)

    def forward(self, batch):
        z_t = self.temporal(batch["temporal"])
        z_s = self.spectral(batch["spectral"])
        z_t_ctx, z_s_ctx, gates = self.coattn(z_t, z_s)
        z_stats = self.stats(batch["stats"])
        z_e2v = self.e2v(batch["e2v"])
        fused = torch.cat([z_t_ctx, z_s_ctx, z_t + z_s, z_stats, z_e2v], dim=1)
        z = self.pre_fusion(fused)
        z_adapt, domain_weights = self.adapter(z)
        return {
            "logits": self.head(z_adapt),
            "temporal_logits": self.temporal_head(z_t),
            "spectral_logits": self.spectral_head(z_s),
            "stats_logits": self.stats_head(z_stats),
            "e2v_logits": self.e2v_head(z_e2v),
            "domain_logits": self.domain_head(z_adapt),
            "domain_weights": domain_weights,
            "coattn_gates": gates,
        }

def predict_ours(model, loader):
    model.eval()
    outputs = defaultdict(list)
    y_list, idx_list = [], []
    with torch.no_grad():
        for batch in loader:
            batch = move_to_device(batch)
            out = model(batch)
            for key in ["logits", "temporal_logits", "spectral_logits", "stats_logits", "e2v_logits", "domain_weights"]:
                outputs[key].append(out[key].detach().cpu().numpy())
            y_list.append(batch["label"].detach().cpu().numpy())
            idx_list.append(batch["row_index"].detach().cpu().numpy())
    merged = {k: np.concatenate(v, axis=0) for k, v in outputs.items()}
    y = np.concatenate(y_list, axis=0)
    idx = np.concatenate(idx_list, axis=0)
    return merged, y, idx

def fit_selective_branch_fusion(val_probs_by_branch, val_y, min_relative=0.80):
    scores = {}
    for name, probs in val_probs_by_branch.items():
        scores[name] = f1_score(val_y, probs.argmax(axis=1), average="macro", zero_division=0)
    vals = np.asarray([scores[k] for k in val_probs_by_branch], dtype=np.float32)
    names = list(val_probs_by_branch.keys())
    max_score = float(vals.max()) if len(vals) else 0.0
    weights = np.where(vals >= min_relative * max_score, vals, 0.0)
    if weights.sum() <= 0:
        weights = np.ones_like(weights)
    weights = weights / weights.sum()
    return dict(zip(names, weights)), scores

def train_ours06f(protocol, split, loaders):
    sample = next(iter(loaders["train"]))
    model = Ours06FDomainAwareMixtureAdapterSER(
        temporal_dim=sample["temporal"].shape[-1],
        stats_dim=sample["stats"].shape[-1],
        e2v_dim=sample["e2v"].shape[-1],
        num_domains=NUM_DOMAINS,
        num_classes=NUM_CLASSES,
    ).to(DEVICE)

    class_weights = class_weights_from_indices(split["train"])
    ce = nn.CrossEntropyLoss(weight=class_weights)
    domain_ce = nn.CrossEntropyLoss()
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    best_state, best_val, best_epoch = None, -1.0, 0
    patience_left = PATIENCE
    history = []
    start = time.time()

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        losses, pred_all, y_all = [], [], []
        for batch in loaders["train"]:
            batch = move_to_device(batch)
            out = model(batch)
            loss_main = ce(out["logits"], batch["label"])
            loss_aux = (
                ce(out["temporal_logits"], batch["label"]) +
                ce(out["spectral_logits"], batch["label"]) +
                ce(out["stats_logits"], batch["label"]) +
                ce(out["e2v_logits"], batch["label"])
            ) * 0.20
            loss_domain = domain_ce(out["domain_logits"], batch["domain"]) * 0.05
            loss = loss_main + loss_aux + loss_domain
            opt.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            opt.step()
            losses.append(float(loss.item()))
            pred_all.extend(out["logits"].argmax(dim=1).detach().cpu().numpy().tolist())
            y_all.extend(batch["label"].detach().cpu().numpy().tolist())

        val_out, val_y, _ = predict_ours(model, loaders["validation"])
        val_pred = val_out["logits"].argmax(axis=1)
        train_m = compute_metrics(np.asarray(y_all), np.asarray(pred_all))
        val_m = compute_metrics(val_y, val_pred)
        history.append({
            "protocol": protocol, "route": "route3_ours06f", "model": "ours_06f_domain_aware_mixture_adapter",
            "epoch": epoch, "train_loss": float(np.mean(losses)),
            **{f"train_{k}": v for k, v in train_m.items()},
            **{f"val_{k}": v for k, v in val_m.items()},
        })
        print(f"{protocol} | route3_ours06f | epoch {epoch:03d} | val_macro_f1={val_m['macro_f1']:.4f}")

        if val_m["macro_f1"] > best_val:
            best_val = val_m["macro_f1"]
            best_epoch = epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            patience_left = PATIENCE
        else:
            patience_left -= 1
            if patience_left <= 0:
                break

    if best_state:
        model.load_state_dict(best_state)

    route = "route3_ours06f"
    model_name = "ours_06f_domain_aware_mixture_adapter"
    pred_paths = {}
    branch_names = ["deep", "temporal", "spectral", "stats", "e2v"]
    branch_logits_map = {
        "deep": "logits",
        "temporal": "temporal_logits",
        "spectral": "spectral_logits",
        "stats": "stats_logits",
        "e2v": "e2v_logits",
    }

    # Fit selective branch weights on validation.
    val_out, val_y, val_idx = predict_ours(model, loaders["validation"])
    val_probs_by_branch = {b: softmax_np(val_out[k]) for b, k in branch_logits_map.items()}
    branch_weights, branch_scores = fit_selective_branch_fusion(
        val_probs_by_branch, val_y,
        min_relative=float(os.getenv("OURS_BRANCH_MIN_RELATIVE_F1", "0.80"))
    )
    pd.DataFrame([
        {"protocol": protocol, "route": route, "branch": b, "val_macro_f1": float(branch_scores[b]), "weight": float(branch_weights[b])}
        for b in branch_names
    ]).to_csv(REPORT_DIR / f"weights_{protocol}_{route}_selective_branch_fusion.csv", index=False)
    print("Our selective branch weights:", branch_weights)

    for split_name in ["train", "validation", "test"]:
        out, y, idx = predict_ours(model, loaders[split_name])
        deep_probs = softmax_np(out["logits"])
        deep_result = compute_metrics(y, deep_probs.argmax(axis=1))
        deep_path = save_prediction_csv(protocol, route, model_name, split_name, idx, deep_probs, y)
        pred_paths[split_name] = deep_path
        GLOBAL_METRICS.append(metrics_row(
            protocol, route, model_name, split_name, deep_result, len(y),
            extra={"best_epoch": best_epoch, "best_val_macro_f1": best_val, "train_time_sec": time.time() - start}
        ))

        fused_probs = np.zeros_like(deep_probs)
        for b, k in branch_logits_map.items():
            fused_probs += float(branch_weights[b]) * softmax_np(out[k])
        fused_result = compute_metrics(y, fused_probs.argmax(axis=1))
        fused_name = "ours_06f_selective_branch_fusion"
        fused_path = save_prediction_csv(protocol, route, fused_name, split_name, idx, fused_probs, y)
        PREDICTION_REGISTRY[(protocol, route)].setdefault(fused_name, {})[split_name] = fused_path
        GLOBAL_METRICS.append(metrics_row(
            protocol, route, fused_name, split_name, fused_result, len(y),
            extra={"branch_weights": json.dumps({k: float(v) for k, v in branch_weights.items()})}
        ))

    model_path = MODEL_DIR / f"{protocol}_{route}_{model_name}.pt".replace("/", "_")
    torch.save({"model_state_dict": model.state_dict(), "protocol": protocol, "route": route, "model": model_name}, model_path)
    for h in history:
        GLOBAL_HISTORY.append(h)
    PREDICTION_REGISTRY[(protocol, route)][model_name] = pred_paths

In [ ]:
# ======================
# Route 3 Training Cell
# ======================

if "3" in RUN_ROUTES:
    for protocol_name, split in protocols.items():
        print("=" * 110)
        print("ROUTE 3 OUR 06F DOMAIN-AWARE MIXTURE ADAPTER |", protocol_name)
        loaders = make_ours_loaders(split)
        train_ours06f(protocol_name, split, loaders)
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
else:
    print("Skip Route 3 because RUN_ROUTES does not contain '3'.")

<a id="leaderboard-and-package-outputs"></a>
# Leaderboard And Package Outputs

Cell này tổng hợp:

- metrics tổng thể.
- per-dataset metrics.
- confusion matrix model tốt nhất từng protocol.
- **3 zip riêng theo route/repo**:
  - `07_route1_Emonity_style_repo_outputs.zip`
  - `07_route2_CNN_LSTM_CLSTM_repo_outputs.zip`
  - `07_route3_Our_06F_model_outputs.zip`
- **1 master zip chứa cả 3 zip route**: `07_SER_Three_Route_Zips_All3.zip`
- **1 zip full output**: `07_SER_Full_Route_Comparison_outputs_full.zip`


In [ ]:
metrics_df = pd.DataFrame(GLOBAL_METRICS)
history_df = pd.DataFrame(GLOBAL_HISTORY)

metrics_path = REPORT_DIR / "07_full_route_comparison_metrics.csv"
history_path = REPORT_DIR / "07_full_route_comparison_history.csv"
metrics_df.to_csv(metrics_path, index=False)
history_df.to_csv(history_path, index=False)

if len(metrics_df):
    test_leaderboard = metrics_df[metrics_df["split"].eq("test")].sort_values(["protocol", "macro_f1"], ascending=[True, False])
    display(test_leaderboard)
else:
    print("No metrics yet. Run at least one route first.")

# Per-dataset metrics from prediction CSVs.
per_dataset_rows = []
for pred_file in PRED_DIR.glob("predictions_*_test.csv"):
    df = pd.read_csv(pred_file)
    stem = pred_file.stem.replace("predictions_", "").replace("_test", "")
    # Safer parsing: protocol and model metadata are also in metrics; this is for report convenience.
    for dataset_name, part in df.groupby("dataset"):
        y = part["true"].map(LABEL_TO_ID).values
        p = part["pred"].map(LABEL_TO_ID).values
        m = compute_metrics(y, p)
        per_dataset_rows.append({
            "prediction_file": pred_file.name,
            "dataset": dataset_name,
            "n": len(part),
            **m
        })
per_dataset_df = pd.DataFrame(per_dataset_rows)
per_dataset_path = REPORT_DIR / "07_full_route_comparison_per_dataset_metrics.csv"
per_dataset_df.to_csv(per_dataset_path, index=False)
if len(per_dataset_df):
    display(per_dataset_df.sort_values(["prediction_file", "dataset"]))

# Figures.
if len(metrics_df):
    plot_df = metrics_df[metrics_df["split"].eq("test")].copy()
    plt.figure(figsize=(16, 7))
    sns.barplot(data=plot_df, x="protocol", y="macro_f1", hue="model")
    plt.xticks(rotation=25, ha="right")
    plt.ylim(0, 1)
    plt.title("Test macro-F1 by protocol and model")
    plt.tight_layout()
    fig_path = FIGURE_DIR / "07_full_route_test_macro_f1_by_protocol_model.png"
    plt.savefig(fig_path, dpi=180)
    plt.show()
    print("Saved:", fig_path)

    plt.figure(figsize=(16, 7))
    sns.barplot(data=plot_df, x="model", y="macro_f1", hue="route")
    plt.xticks(rotation=35, ha="right")
    plt.ylim(0, 1)
    plt.title("Test macro-F1 by model and route")
    plt.tight_layout()
    fig_path = FIGURE_DIR / "07_full_route_test_macro_f1_by_model_route.png"
    plt.savefig(fig_path, dpi=180)
    plt.show()
    print("Saved:", fig_path)

# Confusion matrix for best test model per protocol.
if len(metrics_df):
    test_df = metrics_df[metrics_df["split"].eq("test")].copy()
    for protocol_name, part in test_df.groupby("protocol"):
        best = part.sort_values("macro_f1", ascending=False).iloc[0]
        route = best["route"]
        model = best["model"]
        pred_path_candidates = list(PRED_DIR.glob(f"predictions_{protocol_name}_{route}_{model}_test.csv"))
        if not pred_path_candidates:
            continue
        df = pd.read_csv(pred_path_candidates[0])
        y = df["true"].map(LABEL_TO_ID).values
        p = df["pred"].map(LABEL_TO_ID).values
        cm = confusion_matrix(y, p, labels=list(range(NUM_CLASSES)))
        cm_norm = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)
        plt.figure(figsize=(8, 6))
        sns.heatmap(cm_norm, annot=True, fmt=".2f", xticklabels=COMMON_EMOTIONS, yticklabels=COMMON_EMOTIONS, cmap="Blues")
        plt.title(f"{protocol_name} | best: {route}/{model}")
        plt.xlabel("Predicted")
        plt.ylabel("True")
        plt.tight_layout()
        fig_path = FIGURE_DIR / f"confusion_{protocol_name}_{route}_{model}.png".replace("/", "_")
        plt.savefig(fig_path, dpi=180)
        plt.show()
        print("Saved:", fig_path)

summary = {
    "notebook": "07_Full_Repo_Route_Comparison_SER",
    "objective": "Compare Emonity-style repo, CNN/LSTM/CLSTM-style repo, and our 06F model under shared splits.",
    "processed_root": str(PROCESSED_ROOT),
    "output_dir": str(OUTPUT_DIR),
    "metrics_csv": str(metrics_path),
    "history_csv": str(history_path),
    "per_dataset_csv": str(per_dataset_path),
    "split_manifest_csv": str(split_manifest_path),
    "routes": {
        "1": "Emonity-style: 1D-CNN, 2D-CNN, CNN-BiLSTM, validation-weighted ensemble",
        "2": "CNN/LSTM/CLSTM-style: duration=2.5 offset=0.6, ZCR+RMS+MFCC flatten, noise/pitch/pitch+noise train augmentation",
        "3": "Our 06F: temporal/spectral/stats/e2v-placeholder, co-attention, domain-aware mixture adapter, selective branch fusion",
    },
    "protocols": list(protocols.keys()),
    "labels": COMMON_EMOTIONS,
    "created_at": time.strftime("%Y-%m-%d %H:%M:%S"),
}
summary_path = REPORT_DIR / "07_full_route_comparison_summary.json"
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)


# Package outputs.
# This section creates:
#   1) one FULL zip with every report/prediction/model/split/figure.
#   2) three route-level zips, one per compared repository/model family.
#   3) one master zip that contains the three route-level zips for easy download.

def add_file_to_zip(zf, file_path, base_dir):
    file_path = Path(file_path)
    base_dir = Path(base_dir)
    if file_path.exists() and file_path.is_file():
        try:
            zf.write(file_path, file_path.relative_to(base_dir))
        except ValueError:
            zf.write(file_path, file_path.name)

def add_folder_to_zip(zf, folder, base_dir, predicate=None):
    folder = Path(folder)
    if not folder.exists():
        return
    for file_path in folder.rglob("*"):
        if file_path.is_file() and (predicate is None or predicate(file_path)):
            add_file_to_zip(zf, file_path, base_dir)

def safe_route_token(text):
    return str(text).replace("/", "_").replace("\\", "_").replace(" ", "_")

ROUTE_EXPORTS = {
    "route1_emonity": {
        "route_id": "1",
        "display_name": "Route 1 - Emonity-style repo",
        "zip_name": "07_route1_Emonity_style_repo_outputs.zip",
        "description": "Emonity-style baseline: 1D-CNN, 2D-CNN, CNN-BiLSTM and validation-weighted ensemble.",
    },
    "route2_cnn_lstm_clstm": {
        "route_id": "2",
        "display_name": "Route 2 - CNN/LSTM/CLSTM-style repo",
        "zip_name": "07_route2_CNN_LSTM_CLSTM_repo_outputs.zip",
        "description": "CNN/LSTM/CLSTM-style baseline: duration=2.5, offset=0.6, ZCR+RMS+MFCC flatten, noise/pitch/pitch+noise augmentation.",
    },
    "route3_ours06f": {
        "route_id": "3",
        "display_name": "Route 3 - Our 06F model",
        "zip_name": "07_route3_Our_06F_model_outputs.zip",
        "description": "Our 06F model: temporal/spectral/stats/e2v-placeholder, co-attention, domain-aware mixture adapter and selective branch fusion.",
    },
}

route_zip_rows = []
route_zip_paths = []

for route_key, info in ROUTE_EXPORTS.items():
    route_token = safe_route_token(route_key)
    route_metrics = metrics_df[metrics_df["route"].eq(route_key)].copy() if len(metrics_df) else pd.DataFrame()
    route_history = history_df[history_df["route"].eq(route_key)].copy() if len(history_df) else pd.DataFrame()

    route_metrics_path = REPORT_DIR / f"metrics_{route_token}.csv"
    route_history_path = REPORT_DIR / f"history_{route_token}.csv"
    route_summary_path = REPORT_DIR / f"summary_{route_token}.json"

    route_metrics.to_csv(route_metrics_path, index=False)
    route_history.to_csv(route_history_path, index=False)

    route_test = route_metrics[route_metrics["split"].eq("test")].copy() if len(route_metrics) else pd.DataFrame()
    if len(route_test):
        best_row = route_test.sort_values("macro_f1", ascending=False).iloc[0].to_dict()
    else:
        best_row = None

    route_summary = {
        "route_key": route_key,
        "route_id": info["route_id"],
        "display_name": info["display_name"],
        "description": info["description"],
        "processed_root": str(PROCESSED_ROOT),
        "output_dir": str(OUTPUT_DIR),
        "labels": COMMON_EMOTIONS,
        "protocols": list(protocols.keys()),
        "shared_split_manifest_csv": str(split_manifest_path),
        "n_metric_rows": int(len(route_metrics)),
        "n_history_rows": int(len(route_history)),
        "best_test_row": best_row,
        "created_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    }
    with open(route_summary_path, "w", encoding="utf-8") as f:
        json.dump(route_summary, f, indent=2, ensure_ascii=False)

    zip_path = PROJECT_ROOT / info["zip_name"]
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED, allowZip64=True) as zf:
        # Route-specific reports.
        add_file_to_zip(zf, route_metrics_path, OUTPUT_DIR.parent)
        add_file_to_zip(zf, route_history_path, OUTPUT_DIR.parent)
        add_file_to_zip(zf, route_summary_path, OUTPUT_DIR.parent)

        # Common reports needed for comparing routes fairly.
        add_file_to_zip(zf, metrics_path, OUTPUT_DIR.parent)
        add_file_to_zip(zf, history_path, OUTPUT_DIR.parent)
        add_file_to_zip(zf, per_dataset_path, OUTPUT_DIR.parent)
        add_file_to_zip(zf, summary_path, OUTPUT_DIR.parent)

        # Shared split files: every route zip contains the exact same split manifest.
        add_folder_to_zip(zf, SPLIT_DIR, OUTPUT_DIR.parent)

        # Route-specific prediction files.
        add_folder_to_zip(
            zf,
            PRED_DIR,
            OUTPUT_DIR.parent,
            predicate=lambda p, rk=route_key: f"_{rk}_" in p.name or p.name.startswith(f"predictions_") and f"_{rk}_" in p.name,
        )

        # Route-specific model checkpoints.
        add_folder_to_zip(
            zf,
            MODEL_DIR,
            OUTPUT_DIR.parent,
            predicate=lambda p, rk=route_key: f"_{rk}_" in p.name or rk in p.name,
        )

        # Route-specific figures plus global leaderboard plots.
        add_folder_to_zip(
            zf,
            FIGURE_DIR,
            OUTPUT_DIR.parent,
            predicate=lambda p, rk=route_key: rk in p.name or p.name.startswith("07_full_route_test_"),
        )

    route_zip_paths.append(zip_path)
    route_zip_rows.append({
        "route_id": info["route_id"],
        "route_key": route_key,
        "display_name": info["display_name"],
        "zip_path": str(zip_path),
        "zip_name": info["zip_name"],
        "n_metric_rows": int(len(route_metrics)),
        "n_history_rows": int(len(route_history)),
        "size_mb": round(zip_path.stat().st_size / (1024 * 1024), 3) if zip_path.exists() else 0.0,
    })

route_zip_manifest_df = pd.DataFrame(route_zip_rows)
route_zip_manifest_path = REPORT_DIR / "07_route_zip_manifest.csv"
route_zip_manifest_df.to_csv(route_zip_manifest_path, index=False)
if len(route_zip_manifest_df):
    display(route_zip_manifest_df)

# Master zip containing the 3 route zips.
route_master_zip_path = PROJECT_ROOT / "07_SER_Three_Route_Zips_All3.zip"
with zipfile.ZipFile(route_master_zip_path, "w", compression=zipfile.ZIP_DEFLATED, allowZip64=True) as zf:
    for route_zip_path in route_zip_paths:
        if route_zip_path.exists():
            zf.write(route_zip_path, route_zip_path.name)
    add_file_to_zip(zf, route_zip_manifest_path, OUTPUT_DIR.parent)
    add_file_to_zip(zf, summary_path, OUTPUT_DIR.parent)

# Full zip containing all raw output folders. Keep this for complete reproducibility.
zip_name = "07_SER_Full_Route_Comparison_outputs_full.zip"
zip_path = PROJECT_ROOT / zip_name
package_with_cache = os.getenv("PACKAGE_WITH_CACHE", "0") == "1"
package_dirs = [REPORT_DIR, FIGURE_DIR, MODEL_DIR, PRED_DIR, SPLIT_DIR]
if package_with_cache:
    package_dirs.append(CACHE_DIR)

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED, allowZip64=True) as zf:
    for folder in package_dirs:
        if folder.exists():
            for file_path in folder.rglob("*"):
                if file_path.is_file():
                    zf.write(file_path, file_path.relative_to(OUTPUT_DIR.parent))
    # Include the three route zips and their master zip inside the full package.
    for route_zip_path in route_zip_paths:
        if route_zip_path.exists():
            zf.write(route_zip_path, route_zip_path.name)
    if route_master_zip_path.exists():
        zf.write(route_master_zip_path, route_master_zip_path.name)

print("Saved metrics:", metrics_path)
print("Saved summary:", summary_path)
print("Saved route ZIP manifest:", route_zip_manifest_path)
for row in route_zip_rows:
    print(f"Saved route ZIP {row['route_id']}: {row['zip_path']} ({row['size_mb']} MB)")
print("Saved master route ZIP:", route_master_zip_path)
print("Saved FULL ZIP:", zip_path)
print("PACKAGE_WITH_CACHE:", package_with_cache)
